# Task4 — Ensemble weights 40/50/60%
Train-only cached prediction study. ไม่มีการฝึกหรืออ่าน Test; เก็บ baseline 50/25/25 ถ้าไม่ชนะสม่ำเสมอ

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Task4/models/image_training.py').exists())
OUT=ROOT/'Task4/models/results/ensemble_weights_v1'
scores=pd.read_parquet(OUT/'development_scores.parquet')
result=json.loads((OUT/'result.json').read_text())
display(scores.groupby(['weight','suite']).macro_F1.mean().unstack())

suite,compound,original,single
weight,,,
0.4,0.944253,0.983604,0.980760
0.5,0.946368,0.983914,0.981367
0.6,0.947147,0.984123,0.980757


## Paired deltas ราย fold เทียบ 50%
weight หมายถึงน้ำหนัก EfficientNet; น้ำหนักที่เหลือแบ่ง ResNet สองตัวเท่ากัน

In [2]:
tab=scores.groupby(['fold','weight','suite']).macro_F1.mean().unstack('weight')
tab['delta40']=tab[.4]-tab[.5]
tab['delta60']=tab[.6]-tab[.5]
display(tab)

weight              0.4       0.5       0.6   delta40   delta60
fold suite                                                     
1    compound  0.943430  0.944725  0.944425 -0.001296 -0.000300
     original  0.985331  0.986331  0.986402 -0.001001  0.000071
     single    0.977582  0.977201  0.976574  0.000380 -0.000627
2    compound  0.953160  0.955820  0.958108 -0.002660  0.002288
     original  0.982273  0.983375  0.984100 -0.001102  0.000725
     single    0.985794  0.985600  0.984590  0.000193 -0.001010
3    compound  0.946316  0.948277  0.948868 -0.001961  0.000592
     original  0.985283  0.984449  0.983561  0.000834 -0.000888
     single    0.980616  0.981614  0.979416 -0.000998 -0.002198
4    compound  0.942198  0.942175  0.944519  0.000023  0.002344
     original  0.981869  0.980890  0.982356  0.000979  0.001467
     single    0.979593  0.981004  0.982601 -0.001411  0.001597
5    compound  0.936160  0.940842  0.939817 -0.004682 -0.001025
     original  0.983264  0.984526  0.984195 -0.001262 -0.000331
     single    0.980219  0.981418  0.980606 -0.001199 -0.000812

## เลือกจาก 4 folds ตรวจ fold ที่เหลือ
เลือกด้วย family-balanced stress และกำหนด original/per-class regression guard ไว้ล่วงหน้า ไม่ใช้ audit เลือกน้ำหนัก

ข้อจำกัด: ไม่ใช่ fully nested CV เพราะ base learners ที่สร้าง calibration OOF บางส่วนเคยเห็นข้อมูล outer fold ระหว่างฝึก และ OOF นี้ถูกใช้เลือกโมเดลหลายครั้งแล้ว

In [3]:
display(pd.DataFrame(result['selection']['choices']))
display(result)
display(json.loads((OUT/'protocol.json').read_text()))

,held_fold,selected_weight,calibration_folds
0,1,0.5,"[2, 3, 4, 5]"
1,2,0.5,"[1, 3, 4, 5]"
2,3,0.5,"[1, 2, 4, 5]"
3,4,0.5,"[1, 2, 3, 5]"
4,5,0.5,"[1, 2, 3, 4]"


{'selection': {'choices': [{'held_fold': 1,
    'selected_weight': 0.5,
    'calibration_folds': [2, 3, 4, 5]},
   {'held_fold': 2, 'selected_weight': 0.5, 'calibration_folds': [1, 3, 4, 5]},
   {'held_fold': 3, 'selected_weight': 0.5, 'calibration_folds': [1, 2, 4, 5]},
   {'held_fold': 4, 'selected_weight': 0.5, 'calibration_folds': [1, 2, 3, 5]},
   {'held_fold': 5,
    'selected_weight': 0.5,
    'calibration_folds': [1, 2, 3, 4]}],
  'final_development_weight': 0.5,
  'cross_selected': {'original_gain': 0.0,
   'stress_family_gain': 0.0,
   'stress_wins': 0,
   'worst_class_gain': 0.0,
   'worst_condition_gain': 0.0},
  'fixed_weight': {'original_gain': 0.0,
   'stress_family_gain': 0.0,
   'stress_wins': 0,
   'worst_class_gain': 0.0,
   'worst_condition_gain': 0.0},
  'development_passed': False},
 'audit': {'evaluated': False, 'passed': False},
 'recommend_new_submission': False,
 'keep_weight': 0.5,
 'test_used': False}

{'weights': [0.4, 0.5, 0.6],
 'baseline': 0.5,
 'selection_objective': 'family-balanced development macro F1',
 'selection_original_tolerance': 0.0005,
 'selection_class_tolerance': 0.005,
 'promotion_stress_gain': 0.001,
 'promotion_wins': 4,
 'limitations': ['Reused OOF; base learners behind calibration folds can have seen outer-fold rows, so this is NOT fully nested CV.',
  'Audit conditions have been used before; not an untouched holdout.',
  'No Public-score estimate from similarity or synthetic stress.'],
 'test_used': False,
 'source_sha256': '6b9f25a52752117a30f02985073fe6f66ca36026c56a4f856e81c4ace3124291'}

## ข้อสรุป
- 40% ต่ำกว่า 50% ทั้งสามชุดโดยเฉลี่ย
- 60% ได้ original +0.000209 และ compound +0.000779 แต่ single -0.000610
- เลือกจากอีก 4 folds ได้ 50% ทั้ง 5 ครั้ง
- คง 50/25/25 ไม่สร้าง submission ใหม่ ไม่เปิด audit เพื่อลองเลือกสูตรใหม่ และไม่แตะ Test
- ขั้นต่อไปที่ควรศึกษาเป็นการสร้าง expert ใหม่ เช่น Fourier augmentation จาก training fold เท่านั้น แทนค้นน้ำหนักที่ละเอียดขึ้นบน OOF เดิม

## Reproduce
รันจาก project root: `.venv/bin/python -m Task4.tools.study_ensemble_weights`

การเปิด/Run All notebook นี้อ่านผลที่บันทึกไว้เท่านั้น